# 🤖 AI Multimodal Studio — Google Colab Edition

**No API keys required.** Everything runs locally on Colab's GPU.

| Component | Model | Notes |
|-----------|-------|-------|
| LLM (Chat) | `meta-llama/Llama-3.2-3B-Instruct` | HuggingFace, local |
| TTS | Kokoro-82M | Python 3.12 compatible, local |
| Image Gen | `runwayml/stable-diffusion-v1-5` | HuggingFace, local |

> ⚠️ **Before running**: `Runtime → Change runtime type → T4 GPU`

## Step 1 — Install Dependencies

In [ ]:
# Core ML + UI
!pip install -q gradio diffusers transformers accelerate soundfile sentencepiece protobuf

# Kokoro TTS — Python 3.12 compatible, fully local, no API key
!pip install -q kokoro>=0.9.2 soundfile

# espeak-ng is needed by Kokoro's phonemizer for English
!apt-get install -y -qq espeak-ng

print('✅ All packages installed.')

## Step 2 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU found — image generation will be very slow on CPU.')
    DEVICE = 'cpu'

## Step 3 — Load Models

> ⏳ First run downloads ~7 GB. Subsequent runs use the Colab cache.

In [ ]:
import torch
import re
import numpy as np
import soundfile as sf
import gradio as gr
from threading import Thread

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextIteratorStreamer,
)
from diffusers import AutoPipelineForText2Image
from kokoro import KPipeline

# ── 1. LLM ───────────────────────────────────────────────────────────────────
LLM_MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
print('Loading LLM tokenizer...')
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

print('Loading LLM model (a few minutes on first run)...')
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto',
)
llm_model.eval()
print('✅ LLM loaded.')

# ── 2. TTS — Kokoro-82M ──────────────────────────────────────────────────────
# ~300 MB download. lang_code='a' = American English.
# Runs on CPU fine; GPU not required for TTS.
print('Loading Kokoro TTS...')
tts_pipeline = KPipeline(lang_code='a')
KOKORO_VOICE = 'af_heart'   # warm female voice; others: af_sky, am_adam, am_michael
KOKORO_SR    = 24000
print('✅ TTS loaded.')

# ── 3. Stable Diffusion ───────────────────────────────────────────────────────
print('Loading Stable Diffusion...')
sd_pipe = AutoPipelineForText2Image.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    variant='fp16' if DEVICE == 'cuda' else None,
    low_cpu_mem_usage=True,
).to(DEVICE)
print('✅ Stable Diffusion loaded.')

## Step 4 — Define Functions

In [ ]:
# ── 1. Streaming LLM ─────────────────────────────────────────────────────────

def build_prompt(history, user_input):
    messages = list(history) + [{'role': 'user', 'content': user_input}]
    text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return text, messages


def stream_text(user_input, history):
    prompt_text, messages = build_prompt(history, user_input)
    inputs = llm_tokenizer(
        prompt_text, return_tensors='pt', return_attention_mask=True
    ).to(llm_model.device)

    streamer = TextIteratorStreamer(
        llm_tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    thread = Thread(
        target=llm_model.generate,
        kwargs=dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    )
    thread.start()

    full_response = ''
    for token in streamer:
        full_response += token
        yield messages + [{'role': 'assistant', 'content': full_response}], full_response

    thread.join()


# ── 2. TTS — Kokoro ──────────────────────────────────────────────────────────

def generate_audio(text):
    if not text or not text.strip():
        return None

    audio_path = 'response_audio.wav'
    # Kokoro handles long text natively via its own sentence splitter,
    # but we still cap at ~500 chars to avoid very long pauses before output.
    MAX_CHARS = 500

    # Chunk the text
    sentences = re.split(r'(?<=[.!?]) +', text.replace('\n', ' '))
    chunks, cur = [], ''
    for s in sentences:
        if len(cur) + len(s) <= MAX_CHARS:
            cur += ' ' + s
        else:
            if cur:
                chunks.append(cur.strip())
            cur = s
    if cur:
        chunks.append(cur.strip())

    combined = []
    silence  = np.zeros(int(KOKORO_SR * 0.25))  # 250 ms gap between chunks
    try:
        for chunk in chunks:
            if len(chunk) < 2:
                continue
            # KPipeline yields (graphemes, phonemes, audio_array) tuples
            for _, _, audio in tts_pipeline(chunk, voice=KOKORO_VOICE, speed=1.0):
                combined.append(audio)
            combined.append(silence)

        if not combined:
            return None

        sf.write(audio_path, np.concatenate(combined), KOKORO_SR)
        return audio_path

    except Exception as e:
        print(f'TTS Error: {e}')
        return None


# ── 3. Image-prompt crafter (reuses local LLM) ───────────────────────────────

CARTOON_SYSTEM = (
    'You are an AI that converts chat responses into simple, cartoonish image prompts. '
    'Identify the main subject and describe it in a minimalist 2D vector art style. '
    "Avoid '8k', 'realistic', 'detailed', 'photorealistic'. "
    "Use 'flat colors' and 'simple lines'. Reply with ONLY the image prompt, nothing else.\n\n"
    'Example — Input: A blue owl sat on a branch watching fireflies.\n'
    'Output: A cute blue owl on a tree branch, simple cartoon, flat colors, fireflies background\n'
    'Example — Input: A robot hovered over neon Tokyo streets.\n'
    'Output: A friendly hovering robot, 2D vector art, bright neon colors, minimalist city'
)

def extract_keyword_and_craft_image_prompt(full_text):
    if not full_text:
        return 'a cute simple cat, flat colors, cartoon style'

    messages = [
        {'role': 'system', 'content': CARTOON_SYSTEM},
        {'role': 'user', 'content': f'Convert this to an image prompt: {full_text[:600]}'},
    ]
    prompt_text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(prompt_text, return_tensors='pt').to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs, max_new_tokens=80, do_sample=False, temperature=1.0
        )

    new_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    enhanced = llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    enhanced = re.sub(r'^(Prompt:|Output:)\s*', '', enhanced, flags=re.IGNORECASE)
    print(f'🎨 Image prompt → {enhanced}')
    return enhanced


# ── 4. Image generation ───────────────────────────────────────────────────────

def generate_image(text):
    if not text:
        return None
    try:
        image = sd_pipe(
            prompt=text, num_inference_steps=20, guidance_scale=7.5
        ).images[0]
        image_path = 'generated_image.png'
        image.save(image_path)
        return image_path
    except Exception as e:
        print(f'Image Gen Error: {e}')
        return None


print('✅ All functions defined.')

## Step 5 — Launch Gradio UI

A **public share URL** will appear below — click it to open the app in any browser.

In [ ]:
with gr.Blocks(fill_height=True, title='AI Multimodal Studio') as demo:
    final_text_state   = gr.State('')
    image_prompt_state = gr.State('')

    gr.HTML(
        "<h1 style='text-align:center;font-family:sans-serif;'>🤖 AI Multimodal Studio</h1>"
        "<p style='text-align:center;color:gray;font-family:sans-serif;'>"
        'Chat · Voice · Image — all local, no API keys</p>'
    )

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label='Conversation', height=600, type='messages')
            with gr.Row():
                msg = gr.Textbox(
                    show_label=False, placeholder='Ask me anything…',
                    scale=9, container=False
                )
                submit_btn = gr.Button('Send', variant='primary', scale=1)

        with gr.Column(scale=1):
            image_viewer = gr.Image(label='Image Output', height=400, interactive=False)
            audio_player = gr.Audio(label='Voice Output', interactive=False, autoplay=True)

    def _wire(event):
        """Chain TTS + image generation after LLM streaming finishes."""
        event.then(generate_audio,
                   inputs=[final_text_state], outputs=[audio_player])
        event.then(extract_keyword_and_craft_image_prompt,
                   inputs=[final_text_state], outputs=[image_prompt_state])
        event.then(generate_image,
                   inputs=[image_prompt_state], outputs=[image_viewer])
        event.then(lambda: '', None, [msg])

    btn_event = submit_btn.click(
        fn=stream_text, inputs=[msg, chatbot], outputs=[chatbot, final_text_state]
    )
    submit_btn.click(lambda: '', None, [msg])
    _wire(btn_event)

    sub_event = msg.submit(
        fn=stream_text, inputs=[msg, chatbot], outputs=[chatbot, final_text_state]
    )
    msg.submit(lambda: '', None, [msg])
    _wire(sub_event)


demo.launch(share=True)